<a href="https://colab.research.google.com/github/Vasilisa-Kozlovskaya/SANS_itmo/blob/Lab_1/Sans_modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b Lab_1 https://github.com/Vasilisa-Kozlovskaya/SANS_itmo.git

fatal: destination path 'SANS_itmo' already exists and is not an empty directory.


In [2]:
import os
os.chdir('/content/SANS_itmo')

!ls

 data	     requirements.txt	  src
 README.md   Sans_modules.ipynb  'САНС_ЛБ1 (1).ipynb'


In [3]:
!pip install -r '/content/SANS_itmo/requirements.txt'

In [4]:
pip install --upgrade datasets huggingface_hub

In [5]:
!git pull origin Lab_1

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 2), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 456 bytes | 456.00 KiB/s, done.
From https://github.com/Vasilisa-Kozlovskaya/SANS_itmo
 * branch            Lab_1      -> FETCH_HEAD
   bb96b7a..5e4c5fa  Lab_1      -> origin/Lab_1
Updating bb96b7a..5e4c5fa
Fast-forward
 src/data/datamodule.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


In [6]:
# Импортируем классы и функции из модулей
from src.data.datamodule import CommonCrawlDataModule, WikiTextProcessing
from src.tokenization.tokenizer import CustomTokenizer

In [7]:
WARC_URL = "https://data.commoncrawl.org/crawl-data/CC-NEWS/2025/02/CC-NEWS-20250201012811-00559.warc.gz"

dm = CommonCrawlDataModule(WARC_URL)
dm.prepare_data()
raw_texts = [d['text'] for d in dm.final_data]

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:parsed tree length: 1, wrong data type or not valid HTML
ERROR:trafilatura.core:empty HTML tree: None
ERROR:trafilatura.utils:lxml parsing failed: Document is empty
ERROR:trafilatura.utils:lxml parser bytestring Document is empty
ERROR:trafilatura.core:empty HTML tree: None


Processing 18083 records...


100%|██████████| 18083/18083 [09:48<00:00, 30.72it/s]

Dataset average information density: 0.001459


In [8]:
char_tok = CustomTokenizer(mode="char")
char_tok.train(raw_texts)
print(char_tok.encode("Hello World"))

Char Vocab Size: 1306
[40, 69, 76, 76, 79, 0, 55, 79, 82, 76, 68]


In [9]:
word_tok = CustomTokenizer(mode="word")
word_tok.train(raw_texts)

Word Vocab Size (subset): 7788


In [ ]:
# Обучаем BPE для использования в WikiText
bpe_tok = CustomTokenizer(mode="bpe", vocab_size=5000)
bpe_tok.train(raw_texts)

Training custom BPE tokenizer to vocab_size=5000...


In [ ]:
wiki_processor = WikiTextProcessing(cc_bpe_tokenizer=bpe_tok)
wiki_texts = wiki_processor.process_wikitext()
batches = wiki_processor.create_packed_batches(wiki_texts, block_size=512)